# BERT Yelp Sentiment Analysis

In [ ]:
# Importing neccessary libraries

# Data manipulation & visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Dataset handling
from datasets import Dataset
from imblearn.under_sampling import RandomUnderSampler

# Hugging Face
from transformers import (
    Trainer,
    TrainingArguments,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline
)

# LoRA
from peft import LoraConfig, get_peft_model

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [34]:
# Load the cleaned CSV
reviews_df = pd.read_csv('../data/processed/yelp_reviews_cleaned.csv')
print(f"Loaded {len(reviews_df)} rows from CSV\n")
print(reviews_df.info())

Loaded 99966 rows from CSV

<class 'pandas.DataFrame'>
RangeIndex: 99966 entries, 0 to 99965
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   stars            99966 non-null  int64  
 1   text             99966 non-null  str    
 2   cleaned_text     99962 non-null  str    
 3   sentiment_score  99966 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 88.3 MB
None


In [46]:
# Map the sentiment labels (positive, negative and neutral) to corresponding star ratings in the DataFrame for training
reviews_df['label'] = reviews_df['stars'].replace({
    1: 0, # negative
    2: 0, 
    3: 1, # neutral
    4: 2,
    5: 2  # positive
})

In [ ]:
# Check for class imbalance in the dataset
print(f"Distribution of labels in the dataset:\n{reviews_df['label'].value_counts().sort_index()}")

# Define the data and target features
X = reviews_df[['cleaned_text']] # Convert to a DataFrame as RandomUnderSampler expects 2D features
y = reviews_df['label']

undersampler = RandomUnderSampler(sampling_strategy='not minority', random_state=42) # Resample all classes but the minority class, controlling randomization reproducibility with the magical 42 ("Answer to the Ultimate Question of Life")

X_res, y_res = undersampler.fit_resample(X, y)

print(f"Before undersampling: {Counter(y)}")
print(f"After undersampling: {Counter(y_res)}")

balanced_reviews_df = pd.DataFrame({
    'cleaned_text': X_res['cleaned_text'], # Convert from 2D DataFrame to 1D Series
    'label': y_res
})

print(balanced_reviews_df.sort_index().head())

Distribution of labels in the dataset:
label
0    18902
1    11361
2    69703
Name: count, dtype: int64
Before undersampling: Counter({2: 69703, 0: 18902, 1: 11361})
Before undersampling: Counter({0: 11361, 1: 11361, 2: 11361})
                                        cleaned_text  label
0  decide eat aware going take hours beginning en...      1
2  family diner buffet eclectic assortment large ...      1
5  long term frequent customer establishment went...      0
8  easter instead going lopez lake went los padre...      1
9  party hibachi waitress brought separate sushi ...      1


In [37]:
id2label = {0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"} # A map from index to label
label2id = {"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2} # A map from label to index

In [60]:
# Add mapped label names to the balanced DataFrame for readability
balanced_reviews_df['label_names'] = balanced_reviews_df['label'].replace(id2label)

# Convert the balanced DataFrame to a Hugging Face Dataset object
balanced_reviews_hf = Dataset.from_pandas(balanced_reviews_df)
print(balanced_reviews_hf[:20])

# Save the HuggingFace Dataset to disk
balanced_reviews_hf.save_to_disk('../data/processed/yelp_reviews_balanced_dataset')

{'cleaned_text': ['place mediocre restaurant serves canned beans fake mashed potatoes ranks low decor wornold wont going back', 'ive pizza probably dozen times im almost afraid post review seeing much positive feedback theyve getting money suppose good deal however pizza truly awful ive pizza least half pizza joints ridley area imperial bottom list yes theyre open late pretty much option id rather eat real sand ingredients taste cheap price youll get exactly pay stomach im completely drunk options', 'food location average best beef dip absolutely disgusting beef poor quality undercooked plain horrible wouldnt feed homeless dog', 'ive regular customer past months today decided last visit place service staff cooking staff not polite kept staring friends eating worth mentioning eat restaurant however order make us feel full quickly decided put quantity rice double size sushi left rice decided charge us not polite telling us youre ordering much quantities thats concluded put much rice girl

Saving the dataset (0/1 shards):   0%|          | 0/34083 [00:00<?, ? examples/s]

In [45]:
model = AutoModelForSequenceClassification.from_pretrained('google-bert/bert-base-uncased', num_labels=3, id2label=id2label, label2id=label2id)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
class TextTokenizer:
    def __init__(self, model_name="google-bert/bert-base-uncased"): # Define the default model parameter
        self.tokenizer = AutoTokenizer.from_pretrained(model_name) # Load the tokenizer for the bert-base-uncased model
    
    def tokenize(self, text_batch, max_length=128): # Maximum sequence length set to default
        texts = [str(text) for text in text_batch["cleaned_text"]] # Make sure every input is a string with a list comprehension
        return self.tokenizer(texts,
                              max_length=max_length,
                              padding="max_length", # Ensures equal length
                              truncation=True # Truncates longer sequences to max_length
                              )
    
    # Define a method to convert the tokens back into human-readable format
    def decode(self, token_ids_batch, skip_special_tokens=True):
        return self.tokenizer.batch_decode(token_ids_batch, skip_special_tokens=skip_special_tokens)


# Create an instance (object) of the class
tokenizer_obj = TextTokenizer() # loads the tokenizer for "google-bert/bert-base-uncased" (default)

tokenized_dataset = balanced_reviews_hf.map(tokenizer_obj.tokenize, batched=True)

# Display the first 5 decoded sequences from the tokenized dataset
print(tokenizer_obj.decode(tokenized_dataset['input_ids'][:2]))

Map:   0%|          | 0/34083 [00:00<?, ? examples/s]

In [80]:
# Split 80% train / 20% validation
dataset_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42) # train_test_split automatically shuffles data by default, define test_size (train_size = 1 - test_size), set seed for reproducibility

train_set = dataset_split['train']
test_set = dataset_split['test']

print(f"Train size: {len(train_set)}, Validation size: {len(test_set)}")

Train size: 27266, Validation size: 6817


In [88]:
# Convert the Hugging Face Dataset Objects into PyTorch tensors for compatibility with BERT
train_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_set.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

In [ ]:
lora_config = LoraConfig(
    r=16, # the rank
    lora_alpha=32,
    target_modules=['query', 'value'], # Target only the query and value tensors as they have the strongest impact on model behaviour
    lora_dropout=0.1,
    bias='none',
    task_type='SEQ_CLS'
)

In [ ]:
'''for name, param in model.named_parameters():
    print(name, param.requires_grad)'''

'for name, param in model.named_parameters():\n    print(name, param.requires_grad)'